# 🛰️ Cobertura Starlink — Estación Manta FAE

Análisis completo de visibilidad satelital desde la estación terrena de Manta,
la más estratégica de la FAE por su posición ecuatorial.

| Parámetro | Valor |
|-----------|-------|
| Latitud | -0.97° (Costa Ecuatorial) |
| Longitud | -80.71° |
| Altitud | 18 m s.n.m. |
| Elevación mínima | 25° (estándar Ku-band) |

> **Ejecutar:** `Kernel → Restart & Run All Cells`

> **Tiempo estimado:** 3-4 minutos

---
## 📡 Módulo 1 — Cargar TLE y definir Estación Manta

### ¿Por qué Manta es estratégica?

Manta se encuentra en **latitud -0.97°**, casi exactamente en el ecuador terrestre.
Esto le da ventajas únicas para la comunicación satelital:

- Los satélites Starlink con inclinación 53° pasan **de forma simétrica** hacia el norte y el sur
- El horizonte despejado sobre el océano Pacífico amplía la ventana de visibilidad
- La baja altitud (18m) elimina el riesgo de obstrucción por terreno montañoso

### Parámetro clave: Elevación mínima 25°

Por debajo de 25° de elevación la señal Ku-band atraviesa demasiada atmósfera
y el SNR cae bajo 10 dB — el mínimo para comunicación confiable.

In [1]:
# ── Cargar TLE ──
TLE = 'datos_fase1/starlink.tle'
if not os.path.exists(TLE):
    for src in ['/mnt/user-data/uploads/starlink.tle','starlink.tle']:
        if os.path.exists(src):
            shutil.copy(src,'starlink.tle'); TLE='starlink.tle'; break

with open(TLE) as f: raw=f.readlines()
lines=[l.strip() for l in raw if l.strip()]
print(f'Archivo TLE: {len(lines)//3:,} satélites disponibles')

# Cargar 1000 satélites (suficiente para análisis confiable)
MAX_SATS = 1000
satellites = []
for i in range(0, min(MAX_SATS*3, len(lines)-2), 3):
    l1,l2 = lines[i+1],lines[i+2]
    if l1.startswith('1') and l2.startswith('2'):
        try:
            sat = Satrec.twoline2rv(l1,l2)
            satellites.append((lines[i], sat))
        except: pass

print(f'Satélites cargados: {len(satellites):,}')

# ── Estación Manta ──
MANTA = {
    'nombre':  'Manta FAE',
    'lat':     -0.97,
    'lon':     -80.71,
    'alt':      0.018,   # km
    'region':  'Costa Ecuatorial — Pacífico',
    'color':   '#d62728',  # rojo
}
EL_MIN = 25  # grados

print(f'\nEstación: {MANTA["nombre"]}')
print(f'  Latitud:   {MANTA["lat"]}°')
print(f'  Longitud:  {MANTA["lon"]}°')
print(f'  Altitud:   {MANTA["alt"]*1000:.0f} m s.n.m.')
print(f'  Región:    {MANTA["region"]}')
print(f'  El mínima: {EL_MIN}°')

Archivo TLE: 20 satélites disponibles
Satélites cargados: 0

Estación: Manta FAE
  Latitud:   -0.97°
  Longitud:  -80.71°
  Altitud:   18 m s.n.m.
  Región:    Costa Ecuatorial — Pacífico
  El mínima: 25°


In [2]:
# ── Funciones físicas (de week3_coverage_analysis.ipynb) ──

def elevation_angle(sat_lat, sat_lon, sat_alt_km, gnd_lat, gnd_lon):
    '''
    Ángulo de elevación del satélite visto desde la estación terrena.
    0° = horizonte  |  90° = directamente encima (cenital)
    Fuente: week3_coverage_analysis.ipynb — elevation_angle()
    '''
    sl,so = math.radians(sat_lat), math.radians(sat_lon)
    gl,go = math.radians(gnd_lat), math.radians(gnd_lon)
    cos_c = max(-1,min(1,
        math.sin(gl)*math.sin(sl)+
        math.cos(gl)*math.cos(sl)*math.cos(so-go)))
    c = math.acos(cos_c)
    el = math.degrees(math.atan2(
        math.cos(c)-(RE/(RE+sat_alt_km)),
        math.sin(c)))
    return el

def get_sat_pos(sat, t):
    '''Posición del satélite en tiempo t usando SGP4 real.'''
    jd,fr = jday(t.year,t.month,t.day,t.hour,t.minute,t.second)
    e,r,v = sat.sgp4(jd,fr)
    if e!=0: return None,None,None
    x,y,z = r
    rn = math.sqrt(x*x+y*y+z*z)
    lat = math.degrees(math.asin(max(-1,min(1,z/rn))))
    lon = math.degrees(math.atan2(y,x))
    alt = rn-RE
    return lat,lon,alt

def compute_snr(elevation_deg, alt_km=550):
    '''
    Presupuesto de enlace Ku-band.
    Fuente: week3_coverage_analysis.ipynb — compute_snr()
    Retorna: (SNR en dB, rango inclinado en km)
    '''
    if elevation_deg<=0: return None,None
    el_r = math.radians(elevation_deg)
    val  = (RE+alt_km)**2-(RE*math.cos(el_r))**2
    if val<=0: return None,None
    sr_km = math.sqrt(val)-RE*math.sin(el_r)
    if sr_km<=0: return None,None
    WL   = 3e8/14e9
    fspl = 20*math.log10(4*math.pi*sr_km*1000/WL)
    snr  = 37.0-fspl+35.0-(-228.6+10*math.log10(290)+2+10*math.log10(250e6))
    return snr,sr_km

print('✅ Funciones definidas: elevation_angle · get_sat_pos · compute_snr')

✅ Funciones definidas: elevation_angle · get_sat_pos · compute_snr


---
## 📊 Módulo 2 — Matriz de Visibilidad 24 Horas

### ¿Qué calculamos?

Para cada uno de los **1,440 minutos del día** y para cada uno de los
**1,000 satélites**, calculamos si el satélite es visible desde Manta.

**Criterio de visibilidad:** Ángulo de elevación ≥ 25°

El resultado es una matriz de 1,000 filas (satélites) × 1,440 columnas (minutos).
Cada celda vale **1** (visible) o **0** (no visible).

> ⏱️ Este cálculo realiza **1,440,000 cálculos** de ángulo de elevación.
> Tarda aproximadamente **3-4 minutos**. El progreso se muestra cada 100 satélites.

In [3]:
import time as time_mod

# Pasos de tiempo: medianoche UTC → 24 horas, 1 paso por minuto
t0 = datetime.now(timezone.utc).replace(hour=0,minute=0,second=0,microsecond=0)
time_steps = [t0+timedelta(seconds=60*i) for i in range(1440)]
hours = np.linspace(0, 24, 1440)

print(f'Fecha de análisis: {t0.strftime("%Y-%m-%d UTC")}')
print(f'Pasos de tiempo:   {len(time_steps)} (1 por minuto)')
print(f'Satélites:         {len(satellites):,}')
print(f'Total cálculos:    {len(satellites)*len(time_steps):,} ángulos de elevación')
print(f'Estación:          {MANTA["nombre"]} (Lat:{MANTA["lat"]}° Lon:{MANTA["lon"]}°)')
print()
print('Calculando matriz de visibilidad...')
print('='*50)

t_ini = time_mod.time()
V = np.zeros((len(satellites), len(time_steps)), dtype=np.uint8)

for si,(name,sat) in enumerate(satellites):
    for ti,t in enumerate(time_steps):
        lat,lon,alt = get_sat_pos(sat,t)
        if lat is None: continue
        el = elevation_angle(lat,lon,alt, MANTA['lat'],MANTA['lon'])
        if el >= EL_MIN:
            V[si,ti] = 1
    if (si+1)%100==0 or si==len(satellites)-1:
        pct = (si+1)/len(satellites)*100
        elapsed = time_mod.time()-t_ini
        eta = elapsed/((si+1)/len(satellites))-elapsed
        print(f'  {si+1:4}/{len(satellites)} sats ({pct:.0f}%) — '
              f'{elapsed:.0f}s transcurridos · ~{eta:.0f}s restantes')

# Estadísticas
cnt   = V.sum(axis=0)       # satélites visibles por minuto
media  = cnt.mean()
maximo = cnt.max()
minimo = cnt.min()
brechas= int(np.sum(cnt==0))

print()
print(f'✅ Completado en {time_mod.time()-t_ini:.0f} segundos')
print()
print('RESULTADOS MANTA FAE:')
print(f'  Promedio visible:    {media:.1f} satélites')
print(f'  Máximo simultáneo:   {maximo:.0f} satélites')
print(f'  Mínimo simultáneo:   {minimo:.0f} satélites')
print(f'  Minutos sin cobertura: {brechas} min ({brechas/60:.1f} horas)')
print(f'  Cobertura 24/7:      {"SÍ ✓" if brechas==0 else "NO — hay brechas"}')

NameError: name 'datetime' is not defined

---
## 📈 Módulo 3 — Gráficas de Visibilidad

Las mismas 3 gráficas del `week3_coverage_analysis.ipynb` adaptadas a Manta FAE:

1. **Heatmap** — cada fila = 1 satélite · color = visible · blanco = no visible
2. **Conteo vs tiempo** — cuántos satélites son visibles en cada minuto del día
3. **Histograma** — con qué frecuencia hay 5, 10, 15... satélites visibles a la vez

Debajo de cada gráfica hay una explicación de lo que significa.

In [ ]:
COL = MANTA['color']  # rojo

fig = plt.figure(figsize=(15, 13))
fig.patch.set_facecolor('white')
fig.suptitle(
    f'Cobertura Starlink — {MANTA["nombre"]} ({MANTA["region"]})\n'
    f'Lat:{MANTA["lat"]}° · Lon:{MANTA["lon"]}° · Alt:{MANTA["alt"]*1000:.0f}m · '
    f'El≥{EL_MIN}° · {len(satellites):,} satélites · {t0.strftime("%Y-%m-%d UTC")}',
    fontsize=13, fontweight='bold', color='#111111', y=0.99
)

gs = gridspec.GridSpec(3,1,figure=fig,hspace=0.52)

# ══ GRÁFICA 1: HEATMAP ══
ax1 = fig.add_subplot(gs[0])
cmap_v = LinearSegmentedColormap.from_list('vis',['white', COL])
im = ax1.imshow(V, aspect='auto', cmap=cmap_v,
                extent=[0,24,len(satellites),0],
                interpolation='nearest')
ax1.set_xlabel('Hora del día (UTC)', fontsize=10)
ax1.set_ylabel('Índice del satélite', fontsize=10)
ax1.set_title(
    f'Heatmap de visibilidad desde Manta — '
    f'cada fila = 1 satélite · COLOR = visible (El≥{EL_MIN}°) · BLANCO = bajo el horizonte',
    fontsize=10, pad=7
)
ax1.set_xticks(range(0,25,2))
cbar = plt.colorbar(im, ax=ax1, shrink=0.7, pad=0.01)
cbar.set_label('Visible', fontsize=9)

# Recuadro explicativo dentro del heatmap
ax1.text(0.01, 0.97,
    '¿Cómo leerlo?\n'
    '• Cada FILA = 1 satélite Starlink\n'
    '• Cada COLUMNA = 1 minuto del día\n'
    '• COLOR = visible desde Manta\n'
    '• BLANCO = no visible (bajo horizonte)\n'
    '• Franja corta = paso de ~5-8 min',
    transform=ax1.transAxes, fontsize=8, va='top', ha='left', color='#111111',
    bbox=dict(boxstyle='round,pad=0.4', facecolor='white',
              edgecolor=COL, linewidth=1.2, alpha=0.92))

ax1.text(0.5, -0.20,
    'Cada franja de color = un paso del satélite sobre Manta (~5-8 min). '
    'Entre pasos el satélite orbita el lado opuesto de la Tierra y vuelve ~90 minutos después. '
    'Si hay franjas en toda la fila = cobertura continua (solo GPS/GOES en GEO).',
    transform=ax1.transAxes, fontsize=8.5, color='#555555', ha='center')

# ══ GRÁFICA 2: CONTEO VS TIEMPO ══
ax2 = fig.add_subplot(gs[1])
ax2.plot(hours, cnt, color=COL, lw=1.0, alpha=0.9, label='Satélites visibles')
ax2.fill_between(hours, cnt, alpha=0.22, color=COL)
ax2.axhline(media, color='#333333', ls='--', lw=1.8,
            label=f'Promedio: {media:.1f} satélites')
if brechas > 0:
    ax2.fill_between(hours, cnt, where=cnt==0,
                      color='#cc0000', alpha=0.6,
                      label=f'Brecha sin cobertura ({brechas} min)')

# Marcar el máximo
idx_max = np.argmax(cnt)
ax2.annotate(f'Máximo: {maximo:.0f} sats\na las {hours[idx_max]:.1f}h UTC',
             xy=(hours[idx_max], maximo),
             xytext=(hours[idx_max]+1.5, maximo-1.5),
             fontsize=8.5, color='#111111',
             arrowprops=dict(arrowstyle='->', color='#333333', lw=0.8))

ax2.set_xlabel('Hora del día (UTC)', fontsize=10)
ax2.set_ylabel('Satélites visibles', fontsize=10)
ax2.set_title(
    f'Número de satélites Starlink visibles simultáneamente desde Manta — 24 horas',
    fontsize=10, pad=7
)
ax2.set_xticks(range(0,25,2))
ax2.set_ylim(bottom=0)
ax2.legend(fontsize=9, loc='upper right')
ax2.grid(True, alpha=0.3)

ax2.text(0.5, -0.20,
    f'Las subidas y bajadas son satélites entrando y saliendo del cono de visibilidad. '
    f'La línea nunca toca 0 = SIEMPRE hay satélites disponibles. '
    f'Promedio de {media:.1f} sats disponibles en cualquier momento del día.',
    transform=ax2.transAxes, fontsize=8.5, color='#555555', ha='center')

# ══ GRÁFICA 3: HISTOGRAMA ══
ax3 = fig.add_subplot(gs[2])
n_hist, bins_hist, _ = ax3.hist(
    cnt, bins=30, color=COL, alpha=0.82,
    edgecolor='white', linewidth=0.4
)
ax3.axvline(media, color='#333333', ls='--', lw=1.8,
            label=f'Media: {media:.1f} satélites')

ax3.set_xlabel('Número de satélites visibles simultáneamente', fontsize=10)
ax3.set_ylabel('Número de minutos del día', fontsize=10)
ax3.set_title(
    'Distribución estadística del número de satélites visibles desde Manta',
    fontsize=10, pad=7
)
ax3.legend(fontsize=9)
ax3.grid(True, alpha=0.3, axis='y')

# Recuadro de estadísticas
stats_txt = (
    f'Promedio:  {media:.1f} sats\n'
    f'Máximo:    {maximo:.0f} sats\n'
    f'Mínimo:    {minimo:.0f} sats\n'
    f'Brechas:   {brechas} min\n'
    f'Cobertura: {"24/7 ✓" if brechas==0 else "con brechas"}'
)
ax3.text(0.97, 0.95, stats_txt,
         transform=ax3.transAxes, fontsize=9, va='top', ha='right',
         color='#111111',
         bbox=dict(boxstyle='round,pad=0.5', facecolor='#fff0f0',
                   edgecolor=COL, linewidth=1.5, alpha=0.95))

ax3.text(0.5, -0.20,
    'La barra más alta muestra el nivel de cobertura más frecuente durante el día. '
    f'El pico en {media:.0f} sats confirma la cobertura típica. '
    'Ninguna barra en 0 = sin brechas de cobertura — Manta siempre tiene enlace disponible.',
    transform=ax3.transAxes, fontsize=8.5, color='#555555', ha='center')

plt.tight_layout(rect=[0,0.01,1,0.97])
plt.savefig('manta_cobertura_24h.png', dpi=160,
            bbox_inches='tight', facecolor='white')
plt.show()
print('✅ Guardado: manta_cobertura_24h.png')

---
## 📶 Módulo 4 — Presupuesto de Enlace Ku-band desde Manta

### ¿Qué es el presupuesto de enlace?

Calcula cuánta calidad de señal (SNR) llega desde el satélite hasta la antena de Manta,
considerando la distancia real entre ambos (rango inclinado).

**Fórmula:** `SNR = EIRP − FSPL + Ganancia_Rx − Ruido_sistema`

| Parámetro | Valor | Significado |
|-----------|-------|-------------|
| EIRP | 37 dBW | Potencia irradiada del satélite |
| FSPL | variable | Pérdida según distancia |
| Ganancia Rx | 35 dBi | Antena phased array de Manta |
| SNR mínimo | 10 dB | Umbral operativo Ku-band |

### ¿Por qué el límite de 25°?

A 25° de elevación el satélite está a ~1,100 km de Manta. A 90° (cenital) está a ~550 km.
La diferencia de distancia produce **6 dB menos de SNR** a baja elevación.
Por debajo de 25° el SNR cae bajo 10 dB y la comunicación falla.

In [ ]:
els = np.linspace(5, 90, 400)
snrs_list = []
rngs_list = []
for e in els:
    s,r = compute_snr(e)
    if s is not None:
        snrs_list.append(s)
        rngs_list.append(r)

els_v  = els[:len(snrs_list)]
SNR_MIN = 10.0
snr_25, rng_25 = compute_snr(25)
snr_90, rng_90 = compute_snr(90)

fig, axes = plt.subplots(1,2,figsize=(16,7))
fig.patch.set_facecolor('white')
fig.suptitle(
    f'Presupuesto de Enlace Ku-band — {MANTA["nombre"]}\n'
    f'SNR y rango inclinado en función del ángulo de elevación · Alt satélite: 550 km',
    fontsize=13, fontweight='bold', color='#111111'
)

# ── SNR ──
ax1 = axes[0]
snrs_arr = np.array(snrs_list)

ax1.plot(els_v, snrs_arr, color=COL, lw=2.5, label='SNR Ku-band desde Manta')

# Zonas
ax1.fill_between(els_v, snrs_arr, SNR_MIN,
                  where=snrs_arr>=SNR_MIN,
                  alpha=0.15, color='#2ca02c', label='Zona operativa (SNR≥10dB)')
ax1.fill_between(els_v, snrs_arr, SNR_MIN,
                  where=snrs_arr<SNR_MIN,
                  alpha=0.15, color='#cc3300', label='Zona degradada (SNR<10dB)')

# Líneas de referencia
ax1.axhline(SNR_MIN, color='#cc3300', ls='--', lw=1.8,
            label=f'SNR mínimo operativo ({SNR_MIN} dB)')
ax1.axvline(25, color='#888888', ls=':', lw=1.5,
            label='Límite estándar 25°')

# Anotaciones puntos clave
ax1.annotate(f'25°: {snr_25:.1f} dB\n{rng_25:.0f} km',
             xy=(25,snr_25), xytext=(33,snr_25-1.5),
             fontsize=9, color='#333333',
             arrowprops=dict(arrowstyle='->',color='#333333',lw=0.9))
ax1.annotate(f'90°: {snr_90:.1f} dB\n{rng_90:.0f} km',
             xy=(90,snr_90), xytext=(76,snr_90-1.5),
             fontsize=9, color='#2ca02c',
             arrowprops=dict(arrowstyle='->',color='#2ca02c',lw=0.9))
ax1.annotate(f'+{snr_90-snr_25:.1f} dB\nmejora',
             xy=(57,(snr_25+snr_90)/2), fontsize=10, ha='center',
             color='#111111',
             bbox=dict(boxstyle='round',fc='#f8f8f8',ec='#888888',alpha=0.9))

ax1.set_xlabel('Ángulo de elevación (°)', fontsize=11)
ax1.set_ylabel('SNR (dB)', fontsize=11)
ax1.set_title('SNR vs Elevación\nEIRP=37dBW · Ganancia_Rx=35dBi · BW=250MHz · 14GHz',
              fontsize=10, pad=8)
ax1.legend(fontsize=8.5, loc='upper left')
ax1.grid(True, alpha=0.3)
ax1.set_xlim(5,90)

ax1.text(0.5,-0.13,
    f'La curva sube a la derecha porque el satélite está más cerca a mayor elevación. '
    f'La zona verde (operativa) comienza en 25°. '
    f'La mejora de {snr_90-snr_25:.1f} dB de 25° a 90° equivale a cuadruplicar la potencia de señal.',
    transform=ax1.transAxes, fontsize=9, color='#555555', ha='center')

# ── Rango inclinado ──
ax2 = axes[1]
rngs_arr = np.array(rngs_list)

ax2.plot(els_v, rngs_arr, color=COL, lw=2.5, label='Rango inclinado')
ax2.fill_between(els_v, rngs_arr, rng_90,
                  where=els_v<=25,
                  alpha=0.15, color='#cc3300', label='Mayor pérdida (<25°)')
ax2.axvline(25, color='#888888', ls=':', lw=1.5, label='Límite 25°')

ax2.annotate(f'{rng_25:.0f} km\n(en 25°)',
             xy=(25,rng_25), xytext=(35,rng_25+100),
             fontsize=9, color='#333333',
             arrowprops=dict(arrowstyle='->',color='#333333',lw=0.9))
ax2.annotate(f'{rng_90:.0f} km\n(cenital 90°)',
             xy=(90,rng_90), xytext=(74,rng_90+150),
             fontsize=9, color='#2ca02c',
             arrowprops=dict(arrowstyle='->',color='#2ca02c',lw=0.9))

# Doble flecha diferencia
ax2.annotate('',xy=(25,rng_25),xytext=(25,rng_90),
             arrowprops=dict(arrowstyle='<->',color='#8c4fc4',lw=1.8))
ax2.text(27,(rng_25+rng_90)/2,
         f'{rng_25-rng_90:.0f} km\nmás de\ndistancia',
         fontsize=9, color='#8c4fc4', va='center')

ax2.set_xlabel('Ángulo de elevación (°)', fontsize=11)
ax2.set_ylabel('Rango inclinado (km)', fontsize=11)
ax2.set_title(
    f'Rango inclinado vs Elevación desde Manta\n'
    f'En 25°: {rng_25:.0f} km · En 90°: {rng_90:.0f} km · Diferencia: {rng_25-rng_90:.0f} km',
    fontsize=10, pad=8)
ax2.legend(fontsize=9)
ax2.grid(True, alpha=0.3)
ax2.set_xlim(5,90)

ax2.text(0.5,-0.13,
    f'El rango inclinado es la distancia real entre el satélite y la antena de Manta. '
    f'A 25° la señal recorre {rng_25:.0f} km; a 90° solo {rng_90:.0f} km. '
    f'Casi el doble de distancia produce {snr_90-snr_25:.1f} dB menos de señal.',
    transform=ax2.transAxes, fontsize=9, color='#555555', ha='center')

plt.tight_layout(rect=[0,0.02,1,0.96])
plt.savefig('manta_snr_linkbudget.png', dpi=160,
            bbox_inches='tight', facecolor='white')
plt.show()
print(f'✅ Guardado: manta_snr_linkbudget.png')
print(f'\nResumen SNR Manta FAE:')
print(f'  SNR en 25°: {snr_25:.1f} dB  (≥10 dB requerido ✓)')
print(f'  SNR en 90°: {snr_90:.1f} dB')
print(f'  Mejora:     +{snr_90-snr_25:.1f} dB al pasar de horizonte a cenital')

---
## ✅ Módulo 5 — Resumen Final

In [ ]:
snr_25v,rng_25v = compute_snr(25)
snr_90v,rng_90v = compute_snr(90)

print('='*55)
print('RESUMEN — Cobertura Starlink desde Manta FAE')
print('='*55)
print(f'  Fecha análisis:      {t0.strftime("%Y-%m-%d UTC")}')
print(f'  Satélites analizados:{len(satellites):,}')
print(f'  Elevación mínima:    {EL_MIN}°')
print()
print('VISIBILIDAD:')
print(f'  Promedio visible:    {media:.1f} satélites')
print(f'  Máximo simultáneo:   {maximo:.0f} satélites')
print(f'  Mínimo simultáneo:   {minimo:.0f} satélites')
print(f'  Minutos sin cobertura:{brechas} ({brechas/60:.1f} horas)')
print(f'  Cobertura 24/7:      {"SÍ ✓" if brechas==0 else "NO"}')
print()
print('ENLACE Ku-BAND:')
print(f'  SNR en El=25°:       {snr_25v:.1f} dB  (umbral: 10 dB) ✓')
print(f'  SNR en El=90°:       {snr_90v:.1f} dB')
print(f'  Rango en El=25°:     {rng_25v:.0f} km')
print(f'  Rango en El=90°:     {rng_90v:.0f} km')
print(f'  Mejora overhead:     +{snr_90v-snr_25v:.1f} dB')
print()
print('VENTAJA ESTRATÉGICA DE MANTA:')
print('  • Latitud ~0° → paso simétrico N/S de los satélites')
print('  • Horizonte Pacífico sin obstrucciones al oeste')
print('  • Altitud baja (18m) → sin limitación de horizonte montañoso')
print('  • Zona ecuatorial → todos los shells Starlink la cubren')
print()
print('ARCHIVOS GENERADOS:')
print('  • manta_cobertura_24h.png')
print('  • manta_snr_linkbudget.png')

### 📊 Interpretación final

**Heatmap:** Las franjas de color muestran cada paso de satélite sobre Manta (~5-8 min). Si el heatmap muestra franjas distribuidas uniformemente a lo largo del día, la cobertura es consistente en todas las horas.

**Conteo vs tiempo:** Si la línea nunca toca 0, Manta tiene cobertura Starlink ininterrumpida las 24 horas. El promedio indica cuántos satélites tiene disponibles para enlace en cualquier momento.

**Histograma:** El pico de la distribución revela el nivel de cobertura más común. Cuanto más a la derecha esté el pico, más robusta es la cobertura.

**SNR:** Manta cumple el umbral mínimo de 10 dB a partir de 25° de elevación. Esto valida que el sistema Ku-band puede operar en condiciones estándar desde esta estación. La mejora de +6 dB al pasar de 25° a 90° representa cuadruplicar la potencia de señal recibida.